# Actor-Critic：策略梯度与价值函数的联姻
> 难度标签：高级 | 预计时长：20-30分钟 | 前置知识：无学习经验


> 所属模块：05_强化学习 | 源文件：Actor_Critic.py | 核心功能：双网络架构、TD 更新、优势函数、共享特征

## 概述

Actor-Critic 结合了 REINFORCE（策略梯度）和价值函数的优势。Actor 负责选动作（策略网络），Critic 负责评估状态价值（价值网络）。

REINFORCE 用蒙特卡洛回报 G_t 更新，方差大。Actor-Critic 用 TD 误差（Advantage）替代 G_t，大幅降低方差——代价是引入偏差（bootstrapping）。

脚本用共享底层特征的 Actor-Critic 网络在 CartPole 上训练。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 数学原理

### 1. Actor-Critic 架构

**代码对应**：Actor-Critic 结合策略梯度（Actor）和值函数（Critic）。

**Actor**（策略网络）：$\pi_\theta(a|s)$，输出动作概率分布

**Critic**（值网络）：$V_\phi(s)$，估计状态值函数

### 2. Actor 的更新

用 **TD 误差** $\delta_t$ 代替蒙特卡洛回报 $G_t$：

$$\delta_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$$

Actor 更新：

$$\theta \leftarrow \theta + \alpha_\theta\sum_t\nabla_\theta\ln\pi_\theta(a_t|s_t) \cdot \delta_t$$

### 3. Critic 的更新

Critic 通过最小化 TD 误差的平方更新：

$$\phi \leftarrow \phi - \alpha_\phi\nabla_\phi\sum_t\delta_t^2$$

$$\nabla_\phi\delta_t^2 = 2\delta_t\left(-\nabla_\phi V_\phi(s_t)\right)$$

### 4. 优势函数

TD 误差是优势函数 $A(s, a)$ 的无偏估计：

$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t) \approx r_t + \gamma V(s_{t+1}) - V(s_t) = \delta_t$$

$A > 0$：动作比平均好（鼓励），$A < 0$：动作比平均差（抑制）。

### 5. REINFORCE vs Actor-Critic

| 特性 | REINFORCE | Actor-Critic |
|------|----------|-------------|
| 回报估计 | 蒙特卡洛 $G_t$ | TD 误差 $\delta_t$ |
| 偏差 | 无偏 | 有偏（Critic 不完美） |
| 方差 | 高 | 低 |
| 更新时机 | 轨迹结束后 | 每步更新 |
| 收敛速度 | 慢 | 快 |

### 1. 简单 CartPole 环境

运行 1. 简单 CartPole 环境 的代码，观察算法在该环节的行为。

In [ ]:
class SimpleCartPole:
    def __init__(self):
        self.max_steps = 200
        self.reset()

    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, 4)
        self.steps = 0
        return self.state.copy()

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = 1.0 if action == 1 else -1.0
        x_ddot = (force - 0.0025 * x_dot + 0.001 * np.sin(theta)) / 1.0
        theta_ddot = (0.01 * np.sin(theta) - 0.001 * theta_dot + 0.0001 * force) / 0.1
# --- 赋值/配置 ---
        dt = 0.02
        x_dot += x_ddot * dt
        theta_dot += theta_ddot * dt
        x += x_dot * dt
        theta += theta_dot * dt
# --- 数值计算 ---
        self.state = np.array([x, x_dot, theta, theta_dot])
        self.steps += 1
        done = abs(x) > 2.4 or abs(theta) > 0.21 or self.steps >= self.max_steps
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

### 2. Actor-Critic 网络

运行 2. Actor-Critic 网络 的代码，观察算法在该环节的行为。

In [ ]:
class ActorCritic(nn.Module):
    """共享底层特征提取，分别输出策略（Actor）和价值（Critic）"""
    def __init__(self, state_dim=4, action_dim=2, hidden=64):
        super().__init__()
        self.shared = nn.Sequential(
# --- 继续 ---
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
        )
        # Actor: 输出动作概率
        self.actor = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
            nn.Softmax(dim=-1),
# --- 继续 ---
        )
        # Critic: 输出状态价值
        self.critic = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        features = self.shared(x)
        action_probs = self.actor(features)
        value = self.critic(features)
        return action_probs, value

### 3. Actor-Critic Agent

运行 3. Actor-Critic Agent 的代码，观察算法在该环节的行为。

In [ ]:
class ACAgent:
    def __init__(self, state_dim=4, action_dim=2, lr=3e-4, gamma=0.99):
        self.gamma = gamma
        self.model = ActorCritic(state_dim, action_dim)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def select_action(self, state):
        state_t = torch.FloatTensor(state)
        probs, value = self.model(state_t)
        dist = Categorical(probs)
        action = dist.sample()
# --- 返回结果 ---
        return action.item(), dist.log_prob(action), value

    def update(self, reward, log_prob, value, next_value, done):
        # 计算 TD 误差
        if done:
            target = reward
        else:
            target = reward + self.gamma * next_value.item()

        advantage = target - value.item()

        # Actor 损失：-log_prob × advantage
        actor_loss = -log_prob * advantage
        # Critic 损失：(target - value)²  — value 保持 tensor 参与运算
        target_t = torch.tensor(target, dtype=torch.float32)
        critic_loss = (target_t - value) ** 2

        loss = actor_loss + 0.5 * critic_loss  # 可调 critic 权重

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return actor_loss.item(), critic_loss.item()

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
env = SimpleCartPole()
agent = ACAgent()

print("=== Actor-Critic 训练 ===")
n_episodes = 500
reward_history = []

for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    # 收集一个 episode 的经验
    log_probs = []
    values = []
    rewards = []

    while not done:
        action, log_prob, value = agent.select_action(state)
        next_state, reward, done = env.step(action)

        # 用 TD(0) 在线更新（每步更新，不像 REINFORCE 等 episode 结束）
        next_state_t = torch.FloatTensor(next_state)
        with torch.no_grad():
            _, next_value = agent.model(next_state_t)

        agent.update(reward, log_prob, value, next_value, done)

        state = next_state
        total_reward += reward

    reward_history.append(total_reward)

    if (episode + 1) % 50 == 0:
        avg = np.mean(reward_history[-50:])
        print(f"  Episode {episode+1:>3}: 平均奖励={avg:>6.1f}")

### 5. 测试

运行 5. 测试 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 测试 ===")
test_rewards = []
for _ in range(10):
    state = env.reset()
    total_reward = 0
# --- 赋值/配置 ---
    done = False
    while not done:
        state_t = torch.FloatTensor(state)
        probs, _ = agent.model(state_t)
        action = probs.argmax().item()
# --- 继续 ---
        state, reward, done = env.step(action)
        total_reward += reward
    test_rewards.append(total_reward)
print(f"10 次测试平均奖励: {np.mean(test_rewards):.1f}")

### 6. Actor-Critic 原理

运行 6. Actor-Critic 原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Actor-Critic 原理 ===")
print("Actor（策略网络）: 输出 π(a|s)，决定选什么动作")
print("Critic（价值网络）: 输出 V(s)，评估当前状态有多好")
print("优势函数: A(s,a) = Q(s,a) - V(s) ≈ r + γV(s') - V(s)")
print("  - Actor 更新方向: 增加高优势动作的概率")
# --- 输出结果 ---
print("  - Critic 更新方向: 让 V(s) 更准确地预测回报")

print("\n=== Actor-Critic vs REINFORCE ===")
print("REINFORCE: 蒙特卡洛更新（等 episode 结束），高方差低偏差")
print("Actor-Critic: TD 更新（每步更新），低方差但有偏差（bootstrapping）")
print("  Actor-Critic 收敛更快、更稳定")

print("\n=== Actor-Critic 要点 ===")
print("- 结合了策略梯度和价值函数的优势")
print("- 每步更新（online learning），样本效率更高")
print("- 可扩展到连续动作空间（如 DDPG, SAC）")
print("- 基础版本仍有方差问题 → PPO/IMPALA 等进一步优化")

## 关键代码解释

### 优势函数

```python
advantage = reward + gamma * next_value - value  # TD 误差
```

优势函数 A(s,a) = Q(s,a) - V(s) 衡量"这个动作比平均水平好多少"。正优势增加概率，负优势降低概率。

### 双头网络

```python
self.actor = ...   # 输出动作概率
self.critic = ...  # 输出状态价值 V(s)
```

共享底层特征提取，减少参数量。

## 注意事项

1. **TD 更新 vs MC 更新**：每步更新（低方差有偏差）vs episode 结束更新（高方差无偏差）
2. **学习率敏感**：Actor 和 Critic 可能需要不同学习率
3. **训练不稳定**：基础版本仍有问题，需要 PPO 等改进

## 总结与延伸

以上代码展示了 **Actor Critic** 的完整流程。

**解读要点**：
- 关注 **累计奖励** 随训练轮次的增长趋势
- 观察 **探索率 epsilon** 的衰减过程
- 对比不同策略（greedy vs epsilon-greedy）的表现

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **A2C（Advantage Actor-Critic）**：同步并行版本
- **A3C（Asynchronous Advantage Actor-Critic）**：异步并行版本
- **GAE（Generalized Advantage Estimation）**：平衡偏差和方差的通用优势估计
- **DDPG**：连续动作空间的 Actor-Critic
